1.导入依赖

In [1]:
import os
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from langchain.agents import create_agent

C:\Users\kevin\AppData\Local\Temp\ipykernel_12916\3521569301.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


2.加载环境变量

In [5]:
from dotenv import find_dotenv
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path,override=True)

API_KEY = os.getenv("ARK_API_KEY")
BASE_URL = os.getenv("ARK_BASE_URL")
MODEL = os.getenv("ARK_MODEL")
Tavily_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

3.创建搜索工具

In [6]:
search_tool = TavilySearchResults(max_results=3, api_key=Tavily_API_KEY)

C:\Users\kevin\AppData\Local\Temp\ipykernel_12916\3407506698.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=3, api_key=Tavily_API_KEY)


4.测试搜索工具

In [7]:
result = search_tool.invoke("Shanghai current weather")
result

[{'title': 'Shanghai weather in October 2026 | Shanghai 14 day weather',
  'url': 'https://www.weather25.com/asia/china/shanghai?page=month&month=October',
  'content': '| 25 Sunny 22° /16° | 26 Overcast 23° /17° | 27 Patchy rain possible 22° /16° | 28 Sunny 22° /15° | 29 Partly cloudy 22° /14° | 30 Overcast 23° /16° | 31 Partly cloudy 23° /17° | [...] weather25.com \n\nUnited States England Australia Canada\n\n°F °C\n\nWeather in October 2026\n\nRemove from your favorite locations Add to my locations\n\nShare\n\n1. Home\n2. Asia\n3. weather in ChinaChina\n4. Shanghai\n5. October\n\nLocation was added to My Locations\n\nLocation was removed from My Locations\n\n# Shanghai weather in October 2026\n\nClick on a day for an hourly weather forecast\n\nOct 4\n\nLight rain shower\n\n16.3 mm\n\n20° / 18°Oct 5\n\nOvercast\n\n0 mm\n\n23° / 16°Oct 6\n\nSunny\n\n0 mm\n\n23° / 15°Wednesday\n\nOct 7\n\nSmoky haze\n\n0 mm\n\n26° / 15°Thursday\n\nOct 8\n\nClear\n\n0 mm\n\n25° / 16°Friday\n\nOct 9\n\nP

5.创建天气工具

In [13]:
@tool

def get_weather(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = "https://api.weatherstack.com/current"

    params = {
        "access_key": WEATHERSTACK_API_KEY,
        "query": city,
    }

    response = requests.get(url, params=params, timeout=10)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

6.初始化LLM

In [9]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
    api_key=API_KEY,
    base_url=BASE_URL,
)

7.测试llm

In [12]:
response = llm.invoke("一句话介绍今天上海的天气情况。")
response

AIMessage(content='抱歉，我暂时无法获取上海今天的实时天气，建议查看天气预报或天气应用。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 90, 'total_tokens': 279, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 170, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro-ga-260813', 'system_fingerprint': None, 'id': '021791119128908b531818cd979bd70c39b690ac6f239024bc66e', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10704-fa13-7da2-8acb-0f3d315f7a73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 189, 'total_tokens': 279, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 170}})

8.组装tools

In [14]:
tools = [search_tool, get_weather]

9.创建agent

In [16]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
你是一个天气查询助手。

请遵守以下规则：

1. 如果用户没有提供城市或地点，先询问用户想查询哪个城市。
2. 如果用户没有说明日期，默认查询今天的天气。
3. 查询当前天气时，优先使用天气查询工具获取实时数据。
4. 当天气查询工具无法满足需求，或者用户询问天气相关新闻、预报背景等信息时，可以使用搜索工具。
5. 不要凭空编造实时天气信息。
6. 查询完成后，用中文简洁说明天气情况，包括温度、天气状况和降雨信息。
7. 如果用户的问题与天气无关，也可以正常回答。
"""
)

10.运行并提问

In [19]:
messages = []

print("天气助手已启动，输入 exit 退出。\n")

while True:
    user_input = input().strip()

    if not user_input:
        continue

    print(f"你：{user_input}")

    if user_input.lower() in {"exit", "quit", "退出"}:
        print("天气助手：再见！")
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    final_state = None

    for state in agent.stream(
        {
            "messages": messages
        },
        stream_mode="values",
    ):
        final_state = state

        msg = state["messages"][-1]

        # AI 决定调用工具
        if msg.type == "ai" and getattr(msg, "tool_calls", None):
            for tool_call in msg.tool_calls:
                print(
                    f"[调用工具] {tool_call['name']} "
                    f"{tool_call['args']}"
                )

        # 工具执行结果
        elif msg.type == "tool":
            print(f"[工具返回] {msg.content}")

        # AI 最终回答
        elif msg.type == "ai" and msg.content:
            print(f"天气助手：{msg.content}")

    # 保存完整历史，用于下一轮多轮对话
    if final_state is not None:
        messages = final_state["messages"]

    print()

天气助手已启动，输入 exit 退出。

你：今天上海的天气怎么样
[调用工具] get_weather {'city': '上海'}
[工具返回] City: 上海
Temperature: 18°C
Weather: Overcast 
Humidity: 87%
天气助手：今天上海的天气情况如下：

- **温度**：18°C
- **天气状况**：阴天
- **湿度**：87%

整体比较阴沉，湿度较大，体感可能偏凉。建议外出注意保暖，也可随身携带雨具以防有零星降雨。

你：exit
天气助手：再见！
